# GossipRoboFL — Experiment Analysis

This notebook loads saved experiment results from `results/` and generates all publication-quality plots.

**Before running:** complete at least one experiment run:
```bash
python main.py                                    # gossip baseline
python main.py --mode fedavg                      # FedAvg baseline  
python experiments/ablation_byzantine.py          # robustness sweep
python experiments/ablation_topology.py           # topology sweep
python experiments/ablation_heterogeneity.py      # heterogeneity sweep
```

In [ ]:
import os
import sys
import json
import glob

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Add project root to path
sys.path.insert(0, os.path.dirname(os.getcwd()))

from src.metrics import (
    load_history,
    plot_convergence_curves,
    plot_accuracy_vs_byzantine_fraction,
    plot_communication_overhead,
    plot_non_iid_impact,
    generate_topology_gif,
)

RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 9,
    'lines.linewidth': 2,
})

print('Setup complete.')

## 1. Load All Experiment Logs

In [ ]:
def load_all_histories(results_dir: str) -> dict[str, list[dict]]:
    """Load all *_metrics.json files from results directory."""
    histories = {}
    for path in sorted(glob.glob(os.path.join(results_dir, '*_metrics.json'))):
        name = os.path.basename(path).replace('_metrics.json', '')
        try:
            with open(path) as f:
                histories[name] = json.load(f)
            n_rounds = len(histories[name])
            has_acc = any('honest_global_accuracy' in e for e in histories[name])
            print(f'  {name}: {n_rounds} rounds, accuracy logged={has_acc}')
        except Exception as e:
            print(f'  [skip] {name}: {e}')
    return histories

print(f'Scanning {RESULTS_DIR}...')
histories = load_all_histories(RESULTS_DIR)
print(f'\nLoaded {len(histories)} experiment(s).')

## 2. Convergence Curves — Gossip vs FedAvg

In [ ]:
# Filter for baseline comparisons (gossip mean/ssclip vs fedavg)
baseline_keys = [k for k in histories if any(tag in k for tag in ['fedavg', 'aggmean', 'aggssclip', 'default'])]

if baseline_keys:
    baseline_histories = {k: histories[k] for k in baseline_keys}
    fig = plot_convergence_curves(
        baseline_histories,
        metric='honest_global_accuracy',
        title='Gossip FL vs Centralised FedAvg',
    )
    plt.show()
    fig.savefig(f'{RESULTS_DIR}/nb_convergence_comparison.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
else:
    print('No baseline experiments found. Run main.py first.')

## 3. Byzantine Robustness

In [ ]:
byz_json = os.path.join(RESULTS_DIR, 'ablation_byzantine_results.json')

if os.path.exists(byz_json):
    with open(byz_json) as f:
        byz_raw = json.load(f)
    # Convert string keys back to float
    byz_results = {float(k): v for k, v in byz_raw.items()}

    fig = plot_accuracy_vs_byzantine_fraction(
        byz_results,
        title='Byzantine Robustness: SSClip vs ClippedGossip vs Mean (sign_flip attack)',
    )
    plt.show()
    fig.savefig(f'{RESULTS_DIR}/nb_byzantine_robustness.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Print table
    methods = list(next(iter(byz_results.values())).keys())
    print(f"{'f':>6} | " + ' | '.join(f'{m:>15}' for m in methods))
    print('-' * (10 + 18 * len(methods)))
    for f, accs in sorted(byz_results.items()):
        row = f'{f:>6.1f} | ' + ' | '.join(f"{accs.get(m, float('nan')):>15.4f}" for m in methods)
        print(row)
else:
    print(f'Not found: {byz_json}. Run: python experiments/ablation_byzantine.py')

## 4. Communication Overhead

In [ ]:
# Pick a gossip run to show comm overhead
gossip_keys = [k for k in histories if 'fedavg' not in k and histories[k]]

if gossip_keys:
    key = gossip_keys[0]
    print(f'Showing communication overhead for: {key}')
    fig = plot_communication_overhead(
        histories[key],
    )
    plt.show()
    fig.savefig(f'{RESULTS_DIR}/nb_comm_overhead.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
else:
    print('No gossip experiment found. Run main.py first.')

## 5. Non-IID Impact (Dirichlet Alpha)

In [ ]:
# Load alpha ablation histories
alpha_keys = {0.1: None, 0.5: None, 1.0: None, 10.0: None}

for key, hist in histories.items():
    for alpha in alpha_keys:
        if f'_a{alpha}' in key or f'a{alpha}_' in key:
            alpha_keys[alpha] = hist
            break

alpha_histories = {k: v for k, v in alpha_keys.items() if v is not None}

if alpha_histories:
    fig = plot_non_iid_impact(
        alpha_histories,
        save_path=f'{RESULTS_DIR}/nb_noniid_impact.png',
    )
    plt.show()
    plt.close(fig)
else:
    print('No alpha ablation results found. Run: python experiments/ablation_heterogeneity.py')

## 6. Topology Evolution GIF

In [ ]:
import glob
from IPython.display import Image as IPImage, display

# Check for existing GIF
gif_files = glob.glob(os.path.join(RESULTS_DIR, '*_topology.gif'))
if gif_files:
    print(f'Showing: {gif_files[0]}')
    display(IPImage(filename=gif_files[0]))
else:
    # Try to generate from snapshots
    snap_dir = os.path.join(RESULTS_DIR, 'topology_snaps')
    snaps = sorted(glob.glob(os.path.join(snap_dir, '*.png')))
    if snaps:
        gif_path = os.path.join(RESULTS_DIR, 'topology_evolution.gif')
        generate_topology_gif(snaps, gif_path, fps=3)
        display(IPImage(filename=gif_path))
    else:
        print(f'No topology snapshots found in {snap_dir}.')
        print('Run main.py to generate topology snapshots (saved every update_every rounds).')

## 7. Topology Density Ablation

In [ ]:
topo_json = os.path.join(RESULTS_DIR, 'ablation_topology_results.json')

if os.path.exists(topo_json):
    with open(topo_json) as f:
        topo_results = json.load(f)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    rs = [s['comm_range'] for s in topo_results]
    final_accs = [s['final_accuracy'] for s in topo_results]
    rtt = [s['rounds_to_target'] for s in topo_results]
    gaps = [s['mean_spectral_gap'] for s in topo_results]

    axes[0].plot(rs, final_accs, marker='o', color='#3498db')
    axes[0].set_xlabel('Comm. Range r')
    axes[0].set_ylabel('Final Accuracy')
    axes[0].set_title('Accuracy vs Topology Density')
    axes[0].grid(alpha=0.3)

    axes[1].plot(rs, rtt, marker='s', color='#e74c3c')
    axes[1].set_xlabel('Comm. Range r')
    axes[1].set_ylabel('Rounds to 60% Accuracy')
    axes[1].set_title('Convergence Speed vs Density')
    axes[1].grid(alpha=0.3)

    axes[2].plot(rs, gaps, marker='^', color='#2ecc71')
    axes[2].set_xlabel('Comm. Range r')
    axes[2].set_ylabel('Mean Spectral Gap')
    axes[2].set_title('Algebraic Connectivity vs Density')
    axes[2].grid(alpha=0.3)

    plt.suptitle('Topology Density Ablation (SSClip, no attack)', fontsize=13)
    plt.tight_layout()
    plt.show()
    fig.savefig(f'{RESULTS_DIR}/nb_topology_ablation.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Table
    print(f"{'r':>6} | {'Final Acc':>10} | {'Rounds->60%':>12} | {'Spectral Gap':>12}")
    print('-' * 50)
    for s in topo_results:
        print(f"{s['comm_range']:>6.1f} | {s['final_accuracy']:>10.4f} | {s['rounds_to_target']:>12d} | {s['mean_spectral_gap']:>12.4f}")
else:
    print(f'Not found: {topo_json}. Run: python experiments/ablation_topology.py')

## 8. Summary Dashboard

In [ ]:
# Combined 2x2 dashboard of key results
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])  # Convergence
ax2 = fig.add_subplot(gs[0, 1])  # Byzantine robustness
ax3 = fig.add_subplot(gs[1, 0])  # Comm overhead
ax4 = fig.add_subplot(gs[1, 1])  # Non-IID

colors = plt.cm.tab10(np.linspace(0, 1, 5))

# Panel 1: Convergence curves
plotted = False
for i, (name, hist) in enumerate(list(histories.items())[:5]):
    rounds = [e['round'] for e in hist if 'honest_global_accuracy' in e]
    accs = [e['honest_global_accuracy'] for e in hist if 'honest_global_accuracy' in e]
    if rounds:
        ax1.plot(rounds, accs, label=name[:25], color=colors[i % 5])
        plotted = True
ax1.set_xlabel('Round')
ax1.set_ylabel('Test Accuracy')
ax1.set_title('Convergence Curves')
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=7)
ax1.grid(alpha=0.3)
if not plotted:
    ax1.text(0.5, 0.5, 'No data — run experiments first', ha='center', va='center', transform=ax1.transAxes)

# Panel 2: Byzantine robustness
if os.path.exists(byz_json):
    with open(byz_json) as f:
        byz_raw = json.load(f)
    byz_results = {float(k): v for k, v in byz_raw.items()}
    fractions = sorted(byz_results.keys())
    methods = list(next(iter(byz_results.values())).keys())
    for i, method in enumerate(methods):
        accs = [byz_results[f].get(method, np.nan) for f in fractions]
        ax2.plot([f*100 for f in fractions], accs, marker='o', label=method, color=colors[i])
    ax2.set_xlabel('Byzantine Fraction f (%)')
    ax2.set_ylabel('Final Accuracy')
    ax2.set_title('Byzantine Robustness')
    ax2.set_ylim(0, 1.05)
    ax2.legend()
    ax2.grid(alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'Run ablation_byzantine.py', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('Byzantine Robustness')

# Panel 3: Comm overhead
gossip_keys = [k for k in histories if 'fedavg' not in k and histories[k]]
if gossip_keys:
    hist = histories[gossip_keys[0]]
    rounds = [e['round'] for e in hist if 'honest_global_accuracy' in e]
    accs = [e['honest_global_accuracy'] for e in hist if 'honest_global_accuracy' in e]
    comm = [e.get('cumulative_comm_bytes', 0) / 1e6 for e in hist if 'honest_global_accuracy' in e]
    ax3.plot(rounds, accs, color='#3498db', label='Accuracy')
    ax3b = ax3.twinx()
    ax3b.plot(rounds, comm, color='#e74c3c', linestyle='--', label='Comm (MB)')
    ax3.set_xlabel('Round')
    ax3.set_ylabel('Accuracy', color='#3498db')
    ax3b.set_ylabel('Cumul. Comm (MB)', color='#e74c3c')
    ax3.set_title('Accuracy vs Communication')
    ax3.set_ylim(0, 1.05)
    ax3.grid(alpha=0.3)
else:
    ax3.text(0.5, 0.5, 'Run main.py first', ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('Communication Overhead')

# Panel 4: Non-IID impact (if available)
alpha_histories_filtered = {k: v for k, v in alpha_keys.items() if v is not None}
if alpha_histories_filtered:
    for i, (alpha, hist) in enumerate(sorted(alpha_histories_filtered.items())):
        rounds = [e['round'] for e in hist if 'honest_global_accuracy' in e]
        accs = [e['honest_global_accuracy'] for e in hist if 'honest_global_accuracy' in e]
        ax4.plot(rounds, accs, label=f'alpha={alpha}', color=colors[i])
    ax4.set_xlabel('Round')
    ax4.set_ylabel('Accuracy')
    ax4.set_title('Non-IID Impact (Dirichlet alpha)')
    ax4.legend()
    ax4.set_ylim(0, 1.05)
    ax4.grid(alpha=0.3)
else:
    ax4.text(0.5, 0.5, 'Run ablation_heterogeneity.py', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Non-IID Impact')

fig.suptitle('GossipRoboFL — Results Dashboard', fontsize=15, fontweight='bold')
plt.show()
fig.savefig(f'{RESULTS_DIR}/nb_dashboard.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Dashboard saved to {RESULTS_DIR}/nb_dashboard.png')

## 9. Per-Client Accuracy Distribution

Box plot showing accuracy variance across robots — measures consensus quality.

In [ ]:
# Pick one gossip experiment and plot per-client accuracy over time
if gossip_keys:
    key = gossip_keys[0]
    hist = histories[key]
    
    # Collect per-client accuracy at each eval round
    eval_entries = [e for e in hist if 'per_client_accuracy' in e and e['per_client_accuracy']]
    if eval_entries:
        rounds = [e['round'] for e in eval_entries]
        per_client = [e['per_client_accuracy'] for e in eval_entries]
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Box plots at key rounds
        n_boxes = min(8, len(eval_entries))
        step = max(1, len(eval_entries) // n_boxes)
        box_data = [per_client[i] for i in range(0, len(eval_entries), step)]
        box_labels = [str(rounds[i]) for i in range(0, len(eval_entries), step)]
        
        bp = ax1.boxplot(box_data, labels=box_labels, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('#3498db')
            patch.set_alpha(0.6)
        ax1.set_xlabel('Round')
        ax1.set_ylabel('Client Test Accuracy')
        ax1.set_title(f'Per-Client Accuracy Distribution\n({key})')
        ax1.grid(axis='y', alpha=0.3)
        ax1.set_ylim(0, 1.05)
        
        # Std deviation over rounds (measures consensus)
        stds = [e.get('accuracy_std', np.std(e['per_client_accuracy'])) for e in eval_entries]
        ax2.plot(rounds, stds, color='#e74c3c', linewidth=2)
        ax2.set_xlabel('Round')
        ax2.set_ylabel('Std Dev of Client Accuracies')
        ax2.set_title('Client Consensus (lower std = better consensus)')
        ax2.grid(alpha=0.3)
        ax2.set_ylim(bottom=0)
        
        plt.tight_layout()
        plt.show()
        fig.savefig(f'{RESULTS_DIR}/nb_per_client_accuracy.png', dpi=150, bbox_inches='tight')
        plt.close(fig)
    else:
        print('No per_client_accuracy data found in history.')
else:
    print('No gossip experiment found.')

## 10. Export All Plots

Regenerate all standard plots from saved history files.

In [ ]:
from src.metrics import MetricsTracker

print('Regenerating plots for all experiments...')
for name, hist in histories.items():
    if not hist:
        continue
    # Find byzantine IDs (not stored in history — use empty set for display)
    tracker = MetricsTracker(
        log_dir=RESULTS_DIR,
        run_name=f'nb_{name}',
        num_clients=len(hist[0].get('per_client_accuracy', [1])),
        byzantine_ids=set(),
    )
    tracker.history = hist
    paths = tracker.generate_plots(save_dir=RESULTS_DIR)
    tracker.close()
    if paths:
        print(f'  {name}: {len(paths)} plots')

print('Done!')